# RecipeCatalog example

This notebook shows the prototype `RecipeCatalog`. A catalog combines multiple recipe sources so users can query curated recipes and generated auto recipes through one surface.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

from woodpecker.recipes import DatasetMatcher, FixRef, Recipe
from woodpecker.stores import AutoRecipeStore, JsonRecipeStore, RecipeCatalog
from woodpecker.testing import make_cmip6

Create a CMIP6-like dataset and a small curated recipe store. The curated recipe and the auto store can both point to the same underlying units fix.

In [ ]:
dataset = make_cmip6(overrides={"units": "degC"}, seed=7)

tmp = TemporaryDirectory()
store = JsonRecipeStore(Path(tmp.name) / "recipes.yaml")
store.save_recipe(
    Recipe(
        id="cmip6.curated_units",
        description="Curated CMIP6 units recipe",
        match=DatasetMatcher(dataset_id_patterns=["CMIP6.CMIP.*.Amon.tas.*"]),
        steps=[FixRef(id="woodpecker.normalize_tas_units_to_kelvin")],
    )
)

catalog = RecipeCatalog([store, AutoRecipeStore()])

The catalog lists recipes from all sources and deduplicates by recipe id using source order.

In [ ]:
[recipe.id for recipe in catalog.list_recipes()[:8]]

Lookup queries every source. Here the curated CMIP6 recipe appears first, followed by the generated one-step auto recipe.

In [ ]:
matched_plans = catalog.lookup(dataset)
[recipe.id for recipe in matched_plans]

The same catalog can resolve recipe ids and aliases.

In [ ]:
catalog.get_recipe("cmip6.curated_units")

In [ ]:
tmp.cleanup()